# Phase 1 — Survival Model Analysis & Validation

## Stage III Colon Cancer Overall Survival

This notebook documents the Phase 1 survival-analysis workflow for the oncology survival simulation project.

The primary objective is to develop and validate a multivariable Cox proportional hazards model that can later be compared with a Random Survival Forest (RSF) as alternative parameterizations of baseline patient risk for the downstream microsimulation.

### Primary research question

> Does the choice of survival model — Cox proportional hazards vs. Random Survival Forest — meaningfully change simulated long-term patient outcomes?

### Phase 1 objectives

1. Describe the finalized analysis cohort and survival outcome.
2. Characterize follow-up and censoring.
3. Explore survival patterns using Kaplan-Meier methods.
4. Fit the prespecified multivariable Cox proportional hazards model.
5. Evaluate model coefficients and hazard ratios.
6. Assess the proportional hazards assumption.
7. Apply the predefined diagnostic/remedy hierarchy if necessary.
8. Validate key statistical invariants and model outputs.
9. Freeze and document the final Phase 1 model before beginning the RSF comparison.

## 1. Analysis Scope

### Baseline prognostic modeling

The Cox and RSF models are intended to represent **baseline prognostic risk** for the downstream patient-level microsimulation.

Therefore, the primary model should use information available at or near diagnosis rather than variables that occur after the patient has already entered the disease course.

This distinction is important because the purpose of the model is not to estimate the causal effect of treatment. It is to parameterize patient-level survival risk in a way that can be compared across modeling approaches.

### Primary endpoint

**Overall survival (OS)** is the primary endpoint.

- Time variable: survival time from diagnosis
- Event: death from any cause
- Censoring: alive at last known follow-up

Cause-specific death information is retained for potential sensitivity analysis and quality control but is not used as the primary event definition.

In [1]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from  src.models.models import preprocess_for_modeling

In [ ]:
df = preprocess_for_modeling()

N = df["Time"].count()
mean_followup = df['Time'].mean()
median_followup = df['Time'].median()
event_counts = df['Event'].value_counts().sort_index()
censored_count = event_counts.get(0, 0)
events = event_counts.get(1, 0)

Cleaned data loaded successfully.
Cleaned data shape: (14443, 36)
Recoded 99 -> NaN in: ['Regional nodes examined (1988+)', 'Regional nodes positive (1988+)', 'RX Summ--Surg Prim Site (1998-2022)']


In [ ]:
print(f"Total number of patients: {N}")
print(f"Mean follow-up time: {mean_followup:.2f} months")
print(f"Median follow-up time: {median_followup:.2f} months")
print(f"Number of censored observations: {censored_count}")
print(f"Number of events: {events}")

Total number of patients: 14239
Mean follow-up time: 61.72 months
Median follow-up time: 53.00 months
Number of censored observations: 7832
Number of events: 6407
Event rate: 0.45


## 2. Finalized Analysis Cohort

The analysis cohort was defined before survival modeling.

### Population definition

- Primary site: colon (`C18.0–C18.9`)
- Malignant behavior
- One primary only
- Diagnosis years: 2010–2021
- Exclude autopsy/death-certificate-only reporting sources
- Stage III disease after harmonization across SEER staging eras

### Why 2010–2021?

The study period balances:

- sufficient follow-up for survival analysis,
- consistency across staging systems,
- avoidance of very recent diagnosis years with limited follow-up.

### Stage harmonization

SEER staging fields were extracted as columns rather than filtering on stage during SEER*Stat extraction.

This preserves cases with incomplete or unknown staging information so that stage harmonization and Stage III classification can occur transparently and reproducibly in Python.

### Analysis cohort

* **Cohort Size:** $14,239\text{ patients }$
* **Mean Follow-up:** $61.72\text{ months }$
* **Median Follow-up:** $53.00\text{ months}$
* **Events:** $6407$
* **Censored =** $7832$

## 3. Prespecified Cox Covariates

The primary Cox model uses the following baseline covariates:

| Covariate | Role / Representation |
|---|---|
| Age at diagnosis | Continuous |
| Sex | Binary |
| Stage III substage | Categorical; IIIA reference |
| Marital status | Categorical |
| Race | Categorical |
| Regional nodes examined | Continuous |
| Year of diagnosis | Continuous |

The covariate set was selected based on clinical relevance, population-based survival literature, and the intended role of the model as a baseline prognostic model.

The same baseline covariate set should be used when comparing the Cox model with the RSF model unless a methodological reason for deviation is explicitly documented.

### Treatment variables

Chemotherapy and surgery were extracted from SEER but are not included in the primary baseline prognostic model.

Chemotherapy is excluded because treatment receipt occurs after diagnosis and may introduce treatment-selection bias and immortal-time bias. A naive baseline coefficient for chemotherapy would therefore answer a different question and could contaminate the intended Cox-vs-RSF comparison.

A separate landmark or time-varying treatment analysis could be considered in future work, but that would constitute a different analysis.

## 4. Reference Categories & Encoding

Categorical variables must have explicitly defined reference categories before interpreting Cox coefficients.

### Reference categories

| Covariate | Reference category |
|---|---|
| Sex | `...` |
| Stage III substage | IIIA |
| Marital status | `...` |
| Race | `...` |

### Interpretation

Each categorical hazard ratio represents the estimated relative hazard compared with its defined reference category, holding the other model covariates constant.

For example:

> If IIIB has an HR of 1.40 relative to IIIA, the model estimates approximately 40% higher instantaneous hazard for IIIB compared with IIIA, conditional on the other covariates in the model.

This interpretation assumes the proportional hazards assumption is reasonably satisfied for the covariate.

### Encoding validation

Before fitting the final model, verify that:

- the encoded design matrix contains the expected columns,
- reference categories are actually omitted from dummy encoding,
- coefficient names map unambiguously to the intended covariates,
- no unexpected categories remain,
- the encoded representation matches the documented model specification.